In [ ]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set(style="whitegrid")

data_dir = "../../data_for_prediction/"

file_pattern = str(Path(data_dir) / "*.csv")
files = glob.glob(file_pattern)

targets = ["prec_1d_ahead", "prec_3d_ahead", "prec_7d_ahead"]

MA_WINDOW = 3


def make_baseline_predictions_for_file(csv_path, targets, ma_window=3):
    """
    Compute persistence (t-1) and moving-average baselines
    for one CSV (one location).

    Returns a DataFrame with columns:
      ['location', 'date', 'target', 'model', 'y_true', 'y_pred']
    """
    df = pd.read_csv(csv_path)

    df["date"] = pd.to_datetime(
        df[["YYYY", "MM", "DD"]].rename(columns={
            "YYYY": "year",
            "MM": "month",
            "DD": "day"
        })
    )

    location_id = Path(csv_path).stem

    preds_list = []

    for tgt in targets:
        y_true = df[tgt]
        y_pred = df["prec"] 

        tmp = pd.DataFrame({
            "location": location_id,
            "date": df["date"],
            "target": tgt,
            "model": "persistence",
            "y_true": y_true,
            "y_pred": y_pred
        })
        preds_list.append(tmp)

    ma_series = df["prec"].rolling(window=ma_window,
                                   min_periods=ma_window).mean()

    for tgt in targets:
        y_true = df[tgt]
        y_pred = ma_series

        tmp = pd.DataFrame({
            "location": location_id,
            "date": df["date"],
            "target": tgt,
            "model": f"ma-{ma_window}",
            "y_true": y_true,
            "y_pred": y_pred
        })
        preds_list.append(tmp)

    preds = pd.concat(preds_list, ignore_index=True)

    # Drop rows with missing predictions (start of MA, or missing targets)
    preds = preds.dropna(subset=["y_true", "y_pred"])

    return preds


all_preds = []

for csv_path in files:
    preds_station = make_baseline_predictions_for_file(
        csv_path,
        targets=targets,
        ma_window=MA_WINDOW
    )
    all_preds.append(preds_station)
preds_all = pd.concat(all_preds, ignore_index=True)



# Metrics function (RMSE / MAE)

def compute_metrics(preds_df):
    """
    Compute RMSE and MAE for each (location, target, model).
    """
    def rmse(x):
        return np.sqrt(np.mean((x["y_pred"] - x["y_true"]) ** 2))

    def mae(x):
        return np.mean(np.abs(x["y_pred"] - x["y_true"]))

    metrics = (
        preds_df
        .groupby(["location", "target", "model"], as_index=False)
        .apply(lambda g: pd.Series({
            "RMSE": rmse(g),
            "MAE": mae(g),
            "n": len(g)
        }))
    )
    return metrics

metrics_baseline = compute_metrics(preds_all)

results_dir = Path("../../prediction_results/baseline/")
results_dir.mkdir(parents=True, exist_ok=True)

for model_name, df_model in preds_all.groupby("model"):
    fname = results_dir / f"baseline_predictions_{model_name}.csv"
    df_model.to_csv(fname, index=False)
    print(f"Saved predictions for {model_name} to: {fname}")


metrics_ma3 = metrics_baseline[metrics_baseline["model"] == "ma-3"]
metrics_pers = metrics_baseline[metrics_baseline["model"] == "persistence"]

metrics_ma3_file = results_dir / "baseline_metrics_ma-3.csv"
metrics_pers_file = results_dir / "baseline_metrics_persistence.csv"

metrics_ma3.to_csv(metrics_ma3_file, index=False)
metrics_pers.to_csv(metrics_pers_file, index=False)

print(f"Saved MA-3 metrics to:        {metrics_ma3_file}")
print(f"Saved persistence metrics to: {metrics_pers_file}")

